*NORMALIZACION DE DATOS*


In [16]:
# cargar csv y librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns
from scipy import stats

dataset = '../data/countries.csv'
df = pd.read_csv(dataset, sep=";")
# tamaño del dataset
df.shape


(252, 19)

In [5]:
# nombre de filas y columnas
print('las columnas son: ',df.columns)
# df.head()

las columnas son:  Index(['alpha_2', 'alpha_3', 'area', 'capital', 'continent', 'currency_code',
       'currency_name', 'eqivalent_fips_code', 'fips', 'geoname_id',
       'languages', 'name', 'neighbours', 'numeric', 'phone', 'population',
       'postal_code_format', 'postal_code_regex', 'tld'],
      dtype='str')


In [ ]:

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   alpha_2              251 non-null    str    
 1   alpha_3              252 non-null    str    
 2   area                 252 non-null    float64
 3   capital              246 non-null    str    
 4   continent            210 non-null    str    
 5   currency_code        251 non-null    str    
 6   currency_name        251 non-null    str    
 7   eqivalent_fips_code  1 non-null      str    
 8   fips                 249 non-null    str    
 9   geoname_id           252 non-null    int64  
 10  languages            249 non-null    str    
 11  name                 252 non-null    str    
 12  neighbours           165 non-null    str    
 13  numeric              252 non-null    int64  
 14  phone                247 non-null    str    
 15  population           252 non-null    int64  
 16  p

In [ ]:
# Ver valores nules del dataset
print(df.isnull().sum())

alpha_2                  1
alpha_3                  0
area                     0
capital                  6
continent               42
currency_code            1
currency_name            1
eqivalent_fips_code    251
fips                     3
geoname_id               0
languages                3
name                     0
neighbours              87
numeric                  0
phone                    5
population               0
postal_code_format      98
postal_code_regex      100
tld                      2
dtype: int64


In [8]:
df.dtypes

alpha_2                    str
alpha_3                    str
area                   float64
capital                    str
continent                  str
currency_code              str
currency_name              str
eqivalent_fips_code        str
fips                       str
geoname_id               int64
languages                  str
name                       str
neighbours                 str
numeric                  int64
phone                      str
population               int64
postal_code_format         str
postal_code_regex          str
tld                        str
dtype: object

In [19]:
# PASO 1 — Eliminar columnas irrecuperables (dropna no aplica, directo drop), cuando no tiene sentido 
df = df.drop(columns=["eqivalent_fips_code", "postal_code_regex"])
print("\nPaso 1 — Columnas eliminadas: eqivalent_fips_code, postal_code_regex")


Paso 1 — Columnas eliminadas: eqivalent_fips_code, postal_code_regex


In [ ]:
# PASO 2 — Eliminar filas con valores críticos faltantes (dropna)
antes = len(df)
df = df.dropna(subset=["alpha_2", "name"])
print(f"\n Paso 2 — Filas eliminadas por alpha_2/name nulos: {antes - len(df)}")

# Si un país no tiene código ISO ni nombre, el registro es inutilizable.
# Regla: columnas clave/identificadoras → si faltan, la fila no sirve.


 Paso 2 — Filas eliminadas por alpha_2/name nulos: 1


In [ ]:
# PASO 3 — Imputar con MODA (columnas categóricas con pocos missing)
for col in ["currency_code", "currency_name", "tld", "fips", "phone"]:
    moda = df[col].mode()[0]
    faltaban = df[col].isnull().sum()
    df[col] = df[col].fillna(moda)
    print(f"\n Paso 3 — '{col}': {faltaban} NaN → imputados con moda '{moda}'")

# Son columnas de texto/categoría → no tiene sentido usar media/mediana.
# La moda es el valor más frecuente → imputación conservadora para categorías.
# Regla: variable categórica + pocos missing → imputar con moda.


 Paso 3 — 'currency_code': 1 NaN → imputados con moda 'EUR'

 Paso 3 — 'currency_name': 1 NaN → imputados con moda 'Dollar'

 Paso 3 — 'tld': 2 NaN → imputados con moda '.gp'

 Paso 3 — 'fips': 3 NaN → imputados con moda 'AA'

 Paso 3 — 'phone': 5 NaN → imputados con moda '599'


In [ ]:
# PASO 4 — Imputar 'capital' con "Unknown"
df["capital"] = df["capital"].fillna("Unknown")
print("\n Paso 4 — 'capital': 6 NaN → imputados con 'Unknown'")

# No podemos inventar un nombre real, pero tampoco conviene eliminar esas filas.
# Regla: variable texto sin valor "verdadero" que imputar → marcador explícito.


 Paso 4 — 'capital': 6 NaN → imputados con 'Unknown'


In [ ]:
# PASO 5 — Imputar 'neighbours' y 'languages' con cadena vacía / 0
df["neighbours"] = df["neighbours"].fillna("")
df["languages"]  = df["languages"].fillna("")
print("\n Paso 5 — 'neighbours' y 'languages': NaN → imputados con '' (cero vecinos/idiomas)")
#  El NaN aquí significa "no hay vecinos", no "dato desconocido".
#   Mejor imputar con "" para que el conteo sea 0.
# languages: 3 missing → caso similar, imputamos con cadena vacía.
# Regla: NaN que semánticamente significa "cero/ninguno" → imputar con valor neutro.



 Paso 5 — 'neighbours' y 'languages': NaN → imputados con '' (cero vecinos/idiomas)


In [ ]:
# PASO 6 — Imputar 'continent' con moda (caso especial — muchos missing)
moda_cont = df["continent"].mode()[0]
df["continent"] = df["continent"].fillna(moda_cont)
print(f"\n Paso 6 — 'continent': 42 NaN → imputados con moda '{moda_cont}'")

#  Opciones:
#   a) Eliminar filas → perdemos 42 países (demasiado)
#   b) Imputar con moda → sesgado hacia el continente más frecuente (Africa)
#   c) Imputar manualmente con alpha_2/alpha_3 → lo más correcto
# Aquí hacemos (b) simple; en producción se recomendaría (c).
# Regla: variable importante con missing moderado → imputar con moda
#         y documentar que puede introducir sesgo.


 Paso 6 — 'continent': 42 NaN → imputados con moda 'AF'


In [ ]:
# PASO 7 — Imputar 'postal_code_format' con "N/A"
df["postal_code_format"] = df["postal_code_format"].fillna("N/A")
print("\n Paso 7 — 'postal_code_format': 98 NaN → imputados con 'N/A'")

# 98 missing → muchos países (islas pequeñas, territorios) no usan código postal.
# Igual que neighbours: el NaN significa "no existe", no "desconocido".


 Paso 7 — 'postal_code_format': 98 NaN → imputados con 'N/A'


In [ ]:
# PASO 8 — Eliminar duplicados
antes = len(df)
df = df.drop_duplicates()
print(f"\n Paso 8 — Duplicados eliminados: {antes - len(df)}")

# Buena práctica siempre verificar — en countries no debería haber,
# pero un merge o join previo pudo haberlos generado.


 Paso 8 — Duplicados eliminados: 0


In [ ]:
# PASO 9 — Conversión de tipos
df["population"] = df["population"].astype(int)
df["numeric"]    = df["numeric"].astype(int)
df["area"]       = df["area"].astype(float)
 
text_cols = ["alpha_2","alpha_3","capital","continent","currency_code",
             "currency_name","fips","languages","name","neighbours",
             "phone","postal_code_format","tld"]
for col in text_cols:
    df[col] = df[col].astype(str)
 
print("\n Paso 9 — Tipos de datos ajustados")
print(df.dtypes)

# population y numeric pueden quedar como int (no necesitan decimales)
# area como float (puede tener decimales)
# Las columnas de texto deben ser str, no object


 Paso 9 — Tipos de datos ajustados
alpha_2                   str
alpha_3                   str
area                  float64
capital                   str
continent                 str
currency_code             str
currency_name             str
fips                      str
geoname_id              int64
languages                 str
name                      str
neighbours                str
numeric                 int64
phone                     str
population              int64
postal_code_format        str
tld                       str
dtype: object


In [28]:
# FINAL
print("\n" + "=" * 50)
print("RESUMEN FINAL")
print("=" * 50)
print(f"Shape final : {df.shape}")
print(f"\nMissing restantes:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print("\nDataset limpio ")
 
# Guardar
df.to_csv("countries_clean.csv", index=False)


RESUMEN FINAL
Shape final : (251, 17)

Missing restantes:
Series([], dtype: int64)

Dataset limpio 
